In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from typing import List
import sys
from tyssue import PlanarGeometry
from tyssue.dynamics import effectors, model_factory
import random
from matplotlib.cm import ScalarMappable
import matplotlib.cm as mpl_cm
from tyssue.draw import sheet_view
from collections import defaultdict, deque
from matplotlib.colors import LinearSegmentedColormap, Normalize
from pathlib import Path


import logging
logger = logging.getLogger(__name__)


PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)
import src.auxFunctions as auxFunctions
import src.vertexModel2 as vertexModel2
import src.auxFunctionsExpansion as auxFunctionsExpansion

In [ ]:
### Function to create patches of edges with changed mechanics

def build_edge_adjacency(cellmap):
    """edge_id -> set(neighbour_edge_ids) if share a vertex."""
    v2edges = defaultdict(set)
    for eid, (s, t) in cellmap.edge_df[['srce', 'trgt']].iterrows():
        v2edges[int(s)].add(int(eid))
        v2edges[int(t)].add(int(eid))

    adj = {int(eid): set() for eid in cellmap.edge_df.index}
    for edges in v2edges.values():
        edges = list(edges)
        for i, e1 in enumerate(edges):
            for e2 in edges[i+1:]:
                adj[e1].add(e2)
                adj[e2].add(e1)
    return adj

def make_patches(cellmap, n_patches=50, fraction=0.6, size_jitter=0.25, seed=1):
    # Splits n_target edges across n_patches patches, with size variation between patches controlled by size_jitter
    rng = np.random.default_rng(seed)
    adj = build_edge_adjacency(cellmap)

    edges = np.array(cellmap.edge_df.index, dtype=int)
    n_total = len(edges)
    n_target = int(np.round(float(fraction) * n_total))
    n_target = max(n_target, int(n_patches))

    base = n_target / int(n_patches)
    raw = base * (1 + float(size_jitter) * rng.normal(size=int(n_patches)))
    raw = np.clip(raw, 1, None)
    sizes = np.floor(raw).astype(int)

    while sizes.sum() < n_target:
        sizes[rng.integers(0, len(sizes))] += 1
    while sizes.sum() > n_target:
        i = rng.integers(0, len(sizes))
        if sizes[i] > 1:
            sizes[i] -= 1

    forbidden = set()
    patches = []
    rng.shuffle(edges)
    seed_ptr = 0

    for k in range(int(n_patches)):
        while seed_ptr < len(edges) and int(edges[seed_ptr]) in forbidden:
            seed_ptr += 1
        if seed_ptr >= len(edges):
            break
        seed_edge = int(edges[seed_ptr])
        seed_ptr += 1

        patch = grow_patch(adj, seed_edge, sizes[k], forbidden=forbidden, rng=rng)
        forbidden |= patch
        patches.append(patch)

    return patches

def grow_patch(adj, seed_edge, target_size, forbidden=set(), rng=None):
    # forbidden = edges other patches already claimed, so patches don't overlap
    if rng is None:
        rng = np.random.default_rng()
    patch = set()
    q = deque([int(seed_edge)])

    while q and len(patch) < int(target_size):
        e = int(q.popleft())
        if e in forbidden or e in patch:
            continue
        patch.add(e)
        nbrs = list(adj.get(e, []))
        rng.shuffle(nbrs)
        for n in nbrs:
            if n not in forbidden and n not in patch:
                q.append(int(n))
    return patch


In [ ]:
def add_reverse_half_edges(cellmap, edge_ids):
    """
    Given a list of half-edge IDs,
    return both directions (original + reverse),
    without using MultiIndex (safe for duplicate edges).
    """

    edge_ids = list(edge_ids)
    edge_df = cellmap.edge_df

    both = set(edge_ids)

    for eid in edge_ids:
        if eid not in edge_df.index:
            continue

        s = edge_df.loc[eid, "srce"]
        t = edge_df.loc[eid, "trgt"]

        # find reverse edges explicitly
        rev = edge_df[
            (edge_df["srce"] == t) &
            (edge_df["trgt"] == s)
        ].index

        for r in rev:
            both.add(r)

    return list(both)

def apply_contractility_patches_both_half_edges(cellmap, patches, factor=10.0, edge_col="line_tension"):

    """
    patches = defines number of altered patches
    factor = factor by which the values in edge_col are multipled
    edge_col = choosen between line_tension and longth_elasticity depending on simulation
    """

    high_edges = sorted(set().union(*patches))

    high_edges_both = add_reverse_half_edges(cellmap, high_edges)

    cellmap.edge_df.loc[high_edges_both, edge_col] *= factor

    return high_edges_both

In [ ]:
def relax(cellmap, geom, energyContributions_model, nsteps: int):
    if nsteps <= 0:
        return cellmap
    energyContributions_model.compute_energy(cellmap)
    cellmap_out, _, _, _, _ = vertexModel2.solveEuler(
        cellmap, geom, energyContributions_model, int(nsteps)
    )
    return cellmap_out

def ensure_new_vert_ids(cellmap, next_id):
    vdf = cellmap.vert_df
    if "new_vert_id" not in vdf.columns:
        vdf["new_vert_id"] = np.nan
    missing = vdf["new_vert_id"].isna()
    if missing.any():
        n = int(missing.sum())
        vdf.loc[missing, "new_vert_id"] = np.arange(next_id, next_id + n, dtype=int)
        next_id += n
    return next_id

def save_visualisation_with_new_vertices(cellmap, geom, current_step, output_dir, divisions_count, collapses_count, col="length_elasticity"):
    """
    Create and save visualization with new vertices (birth_step == current_step) highlighted in red.
    """
    new_vert_ids = cellmap.vert_df[cellmap.vert_df["birth_step"] == current_step].index.tolist()

    fig, ax = plot_coloured_edges_with_new_vertices(
        cellmap,
        geom,
        new_vert_ids=new_vert_ids,
        col=col,
        threshold=None,
        width=3.0,
        show_axes=True,
        xlim=(-40, 80),
        ylim=(-40, 80),
        tick_spacing=None,
        new_vert_colour="red",
        new_vert_size=80,
        figsize=(10, 10),
        title=None,
        show_old_vertices=False,
    )
    
    plt.savefig(os.path.join(output_dir, f"progress_{current_step:06d}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Visualization saved at step {current_step}")


In [ ]:
def plot_coloured_edges_with_new_vertices(
    cellmap,
    geom,
    new_vert_ids=None,
    col="length_elasticity",
    threshold=None,
    width=3.0,
    show_axes=True,
    xlim=(-20, 60),
    ylim=(-20, 60),
    tick_spacing=None,
    new_vert_colour="red",
    new_vert_size=80,
    figsize=(10, 10),
    title=None,
    show_old_vertices=False,
):
    """
    Plot cellmap with edges coloured by a property (length_elasticity or line_tension)
    and highlight new vertices in red.

    Parameters:
    -----------
    cellmap : Tyssue cellmap
    geom : Geometry object
    new_vert_ids : list
        List of vertex indices to highlight in red (new division vertices)
    col : str
        Column name for edge colouring — "length_elasticity" or "line_tension"
    threshold : float or None
        If None, continuous colouring. If float, binary colouring.
    width : float
        Edge width
    show_axes : bool
        Show axes labels
    xlim, ylim : tuple
        Axis limits
    tick_spacing : float or None
        Spacing between ticks
    new_vert_colour : str
        Colour for new vertices
    new_vert_size : float
        Size of new vertices
    figsize : tuple
        Figure size
    title : str or None
        Plot title
    show_old_vertices : bool, default=False
        Whether to show non-divided vertices (if False, alpha=0)

    Returns:
    --------
    fig, ax : matplotlib figure and axes
    """

    if col == "length_elasticity":
        low_colour, high_colour = "mediumturquoise", "orchid"
        default_threshold = 250
    elif col == "line_tension":
        low_colour, high_colour = "mediumturquoise", "orange"
        default_threshold = 160
    else:
        raise ValueError(f"No colour scheme defined for col='{col}'")

    if threshold is None:
        threshold = default_threshold

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    coords = cellmap.coords[:2]
    edge_df = cellmap.edge_df
    vert_df = cellmap.vert_df

    values = edge_df[col].values

    # Determining colours for each edge
    if threshold is None:
        vmin, vmax = values.min(), values.max()
        norm = Normalize(vmin, vmax) if vmax > vmin else Normalize(0, 1)
        cmap = LinearSegmentedColormap.from_list('custom', [low_colour, high_colour])
        edge_colours = cmap(norm(values))
    else:
        edge_colours = [high_colour if val >= threshold else low_colour for val in values]

    for idx, row in edge_df.iterrows():
        srce = row['srce']
        trgt = row['trgt']

        x1 = vert_df.loc[srce, coords[0]]
        y1 = vert_df.loc[srce, coords[1]]
        x2 = vert_df.loc[trgt, coords[0]]
        y2 = vert_df.loc[trgt, coords[1]]

        ax.plot(
            [x1, x2], [y1, y2],
            color=edge_colours[idx],
            linewidth=width,
            solid_capstyle='round',
            zorder=1,
            alpha=0.9,
        )

    x = vert_df[coords[0]].values
    y = vert_df[coords[1]].values
    vert_alpha = 0.7 if show_old_vertices else 0.0

    ax.scatter(
        x, y,
        color='lightgrey',
        s=30,
        alpha=vert_alpha,
        zorder=2,
        edgecolor='darkgrey',
        linewidth=0.5,
    )

    if new_vert_ids is not None and len(new_vert_ids) > 0:
        existing_verts = set(vert_df.index)
        valid_new_verts = [v for v in new_vert_ids if v in existing_verts]

        if len(valid_new_verts) > 0:
            new_x = vert_df.loc[valid_new_verts, coords[0]].values
            new_y = vert_df.loc[valid_new_verts, coords[1]].values

            ax.scatter(
                new_x, new_y,
                color=new_vert_colour,
                s=new_vert_size,
                alpha=1.0,
                zorder=3,
                edgecolor='darkred',
                linewidth=1.5,
            )

            print(f"Highlighted {len(valid_new_verts)} new vertices in {new_vert_colour}")

    ax.set_aspect('equal')

    if show_axes:
        ax.set_xlabel("X", fontsize=14)
        ax.set_ylabel("Y", fontsize=14)
        if tick_spacing is not None:
            ax.set_xticks(np.arange(xlim[0], xlim[1] + tick_spacing, tick_spacing))
            ax.set_yticks(np.arange(ylim[0], ylim[1] + tick_spacing, tick_spacing))
    else:
        ax.set_xticks([])
        ax.set_yticks([])

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    if title is not None:
        ax.set_title(title, fontsize=14)

    plt.tight_layout()

    return fig, ax

### Simulation of expasnion with altered tissue mechanics

In [ ]:
def run_expansion(
    simulation_number,
    checkpoint_dir="../../Expansion_checkpoints",
    checkpoint_step=3500,
    n_patches=50, 
    fraction=0.1,
    seed=1,
    factor=10,
    edge_col="length_elasticity",
    total_steps=12000,
    steps_per_cycle=500,
    relax_steps=500,
    edge_sum_threshold=3.5,
    division_distance=0.01,
    collapse_fraction=0.0005,
    relax_after_collapse=10,
    output_directory=None,
):
    sim_names = {1: "one", 2: "two", 3: "three"}

    if output_directory is None:
        output_directory = f"sim_{simulation_number}_{n_patches}patches_{int(fraction*100)}pct_{edge_col}_{factor}x"

    def ensure_new_vert_ids(cellmap, next_id):
        """Assign unique IDs to any new vertices (from collapses etc)."""
        vdf = cellmap.vert_df
        if "new_vert_id" not in vdf.columns:
            vdf["new_vert_id"] = np.nan
        missing = vdf["new_vert_id"].isna()
        if missing.any():
            n = int(missing.sum())
            vdf.loc[missing, "new_vert_id"] = np.arange(next_id, next_id + n, dtype=int)
            next_id += n
        return next_id

    with open(
        os.path.join(checkpoint_dir, f"simulation_{sim_names[simulation_number]}_checkpoint_{checkpoint_step:06d}.pkl"),
        "rb"
    ) as f:
        cellmap = pickle.load(f)

    
    required_cols = ["new_vert_id", "parent_vert_id", "birth_step", "divided_step"]
    for col in required_cols:
        if col not in cellmap.vert_df.columns:
            raise ValueError(f"Column '{col}' missing from vert_df — this should NOT happen!")
    next_new_vert_id = int(cellmap.vert_df["new_vert_id"].max()) + 1

    geom = PlanarGeometry
    energyContributions_model = model_factory([
        effectors.FaceAreaElasticity,
        effectors.LineTension,
        effectors.LengthElasticity,
    ])

    patches = make_patches(cellmap, n_patches=n_patches, fraction=fraction, seed=seed)
    apply_contractility_patches_both_half_edges(cellmap, patches, factor=factor, edge_col=edge_col)


    cellmap.reset_topo()

    starting_step = checkpoint_step

    # Main loop of simulation

    print(f"\n{'='*60}")
    print(f"Continuing expansion from step {starting_step}")
    print(f"Total additional steps: {total_steps} (to step {starting_step + total_steps})")
    print(f"Saving every {steps_per_cycle} steps")
    print(f"{'='*60}\n")

    # Creating output directory
    os.makedirs(output_directory, exist_ok=True)

    # Statistics
    division_stats = []
    collapse_stats = []

    # Saving initial state at step 3500
    initial_step = starting_step
    with open(os.path.join(output_directory, f"checkpoint_{initial_step:06d}.pkl"), "wb") as f:
        pickle.dump(cellmap, f)
    print(f"Initial checkpoint saved at step {initial_step}")

    # Save initial visualization using the new style
    # Find vertices born at step 3500 (should be from previous simulation)
    new_vert_ids_initial = cellmap.vert_df[cellmap.vert_df["birth_step"] == initial_step].index.tolist()
    fig, ax = plot_coloured_edges_with_new_vertices(
        cellmap,
        geom,
        new_vert_ids=new_vert_ids_initial,
        col=edge_col,
        threshold=None,
        width=3.0,
        show_axes=True,
        xlim=(-40, 80),
        ylim=(-40, 80),
        tick_spacing=None,
        new_vert_colour="red",
        new_vert_size=80,
        figsize=(10, 10),
        title=None,
        show_old_vertices=False,
    )
    
    plt.savefig(os.path.join(output_directory, f"progress_{initial_step:06d}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Initial visualization saved at step {initial_step}")

    cellmap.face_df['prefered_area'] *= 1.2

    # Main expansion loop
    for cycle in range(1, int(total_steps / steps_per_cycle) + 1):
        current_step = starting_step + cycle * steps_per_cycle
        print(f"\n{'='*60}")
        print(f"Cycle {cycle}: Step {current_step}")
        print(f"{'='*60}")

        # --------------------------------------------------------
        # Step 1: Identify inside vertices
        # --------------------------------------------------------
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, 1)

        # --------------------------------------------------------
        # Step 1b: COLLAPSES (apoptosis)
        # --------------------------------------------------------
        current_collapses = 0

        valid_inside_edges = [e for e in inside_edges if e in cellmap.edge_df.index]
        n_inside = len(valid_inside_edges)
        target_collapse = int(np.floor(collapse_fraction * n_inside))
        target_collapse = max(0, min(target_collapse, n_inside))

        if target_collapse > 0:
            edges_to_collapse = np.random.choice(valid_inside_edges, size=target_collapse, replace=False)
            for edge in edges_to_collapse:
                try:
                    cellmap = auxFunctionsExpansion.collapse_single_edge_expansion(
                        cellmap, geom, energyContributions_model, edge
                    )
                    current_collapses += 1
                except Exception as e:
                    print(f"  Warning: collapse failed for edge {edge}: {e}")

        print(f"  Collapses: {current_collapses}/{target_collapse}")

        cellmap.reset_index()
        cellmap.reset_topo()
        next_new_vert_id = ensure_new_vert_ids(cellmap, next_new_vert_id)

        if relax_after_collapse > 0:
            cellmap = relax(cellmap, geom, energyContributions_model, relax_after_collapse)
            cellmap.reset_index()
            cellmap.reset_topo()

        # Re-identify boundaries after collapses
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, 1)

        # Step 2: Perform divisions
        vertices_to_divide = auxFunctionsExpansion.identify_vertices_to_divide(
            cellmap, inside_vertices, edge_sum_threshold)
        
        print(f"Found {len(vertices_to_divide)} vertices above threshold")

        current_divisions = 0

        if len(vertices_to_divide) > 0:
            divided_count = 0
            
            for v in vertices_to_divide:
                try:
                    cellmap, parent_vert, new_vert, new_edge, opp_edge = auxFunctionsExpansion.split_vertex_expansion(
                        cellmap, v, geom, energyContributions_model, division_distance, retry_attempts=3
                    )
                    
                    if new_vert is not None:
                        # Record lineage with existing columns
                        parent_id = int(cellmap.vert_df.loc[parent_vert, "new_vert_id"])
                        cellmap.vert_df.loc[parent_vert, "divided_step"] = current_step
                        cellmap.vert_df.loc[new_vert, "new_vert_id"] = next_new_vert_id
                        cellmap.vert_df.loc[new_vert, "parent_vert_id"] = parent_id
                        cellmap.vert_df.loc[new_vert, "birth_step"] = current_step
                        next_new_vert_id += 1
                        divided_count += 1
                        
                except Exception as e:
                    print(f"  Warning: division failed for vertex {v}: {e}")
            
            current_divisions = divided_count
            print(f"Performed {current_divisions} divisions")

            cellmap.reset_index()
            cellmap.reset_topo()

        division_stats.append(current_divisions)
        collapse_stats.append(current_collapses)


        # Step 3: Save post-division checkpoint

        with open(os.path.join(output_directory, f"checkpoint_{current_step:06d}_postdiv.pkl"), "wb") as f:
            pickle.dump(cellmap, f)
        print(f"Post-division checkpoint saved at step {current_step}")

        # Step 4: Energy minimization / relaxation

        print(f"Relaxing for {relax_steps} steps...")
        energyContributions_model.compute_energy(cellmap)
        [cellmap, geom, energyContributions_model, history_new, solver1] = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, relax_steps
        )
        print(f"Relaxation complete.")

        # Step 5: Save checkpoint
        with open(os.path.join(output_directory, f"checkpoint_{current_step:06d}.pkl"), "wb") as f:
            pickle.dump(cellmap, f)
        print(f"Checkpoint saved at step {current_step}")

        # Step 6: Save visualization using plot_coloured_edges_with_new_vertices
        save_visualisation_with_new_vertices(
            cellmap, geom, current_step, output_directory, current_divisions, current_collapses, col=edge_col
        )           

        # Step 7: Update division stats
        np.savetxt(os.path.join(output_directory, "division_stats.txt"), division_stats)
        np.savetxt(os.path.join(output_directory, "collapse_stats.txt"), collapse_stats)

    # 6. FINAL SUMMARY

    print(f"\n{'='*60}")
    print("EXPANSION COMPLETE!")
    print(f"Total divisions: {sum(division_stats)}")
    print(f"Total collapses: {sum(collapse_stats)}")
    print(f"Final vertex count: {len(cellmap.vert_df)}")
    print(f"Final edge count: {len(cellmap.edge_df)}")
    print(f"Output saved to: {output_directory}")
    print(f"{'='*60}")

    # Plot division and collapse statistics
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    steps = [starting_step + i * steps_per_cycle for i in range(1, len(division_stats) + 1)]

    ax1.plot(steps, division_stats, 'bo-', markersize=8)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Divisions per cycle')
    ax1.set_title('Divisions over time')
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps, collapse_stats, 'ro-', markersize=8)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Collapses per cycle')
    ax2.set_title('Collapses over time')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(output_directory, "division_collapse_stats.png"), dpi=150)
    plt.show()

    # Save final visualization
    save_visualisation_with_new_vertices(
        cellmap, geom, current_step, output_directory, current_divisions, current_collapses, col=edge_col
    )

    # Save final cellmap
    with open(os.path.join(output_directory, "checkpoint_final.pkl"), "wb") as f:
        pickle.dump(cellmap, f)
    print(f"Final checkpoint saved to: {output_directory}/checkpoint_final.pkl")

    return cellmap

In [ ]:
cellmap = run_expansion(
    simulation_number=2,
    checkpoint_dir="../../Expansion_checkpoints",
    checkpoint_step=3500,
    n_patches=10, 
    fraction=0.5,
    seed=1,
    factor=3,
    edge_col="length_elasticity",
    total_steps=15000,
    steps_per_cycle=500,
    relax_steps=500,
    edge_sum_threshold=3.5,
    division_distance=0.01,
    collapse_fraction=0.0005,
    relax_after_collapse=10,
    output_directory=None)